# Chapter 14: RAG Evaluation End-to-End
## Topic 5: A/B Testing Retrieval Strategies

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every earlier topic in this chapter — RAGAS's metrics (Topic 1), a real eval set (Topic 2), running that eval set through this project's actual pipeline (Topic 3), tracing individual requests (Topic 4) — has built the infrastructure this closing topic finally puts to its intended use: comparing two different retrieval strategies against each other, using a controlled, evidence-based methodology, rather than deciding between them based on intuition.
- This topic is the direct, natural culmination of a controlled-comparison discipline this notebook series has established repeatedly, in slightly different forms, across many earlier chapters: Chapter 7 Topic 9's retrieval evaluation harness, Chapter 9 Topic 10's chunking-strategy comparison, Chapter 10 Topic 4's schema-wording comparison, Chapter 10 Topic 7's regression-testing baseline comparison — every one of these was, at its core, an A/B test: hold everything constant except one variable, measure the same metrics on both configurations, and let the measured difference (not intuition) determine which configuration is actually better.
- The specific new contribution this topic makes: applying this same, already-familiar controlled-comparison discipline specifically to *retrieval strategy* choices, using RAGAS's context precision and context recall metrics (Topic 1) as the actual comparison signal, computed via Topic 3's methodology, on Topic 2's real eval set — bringing every piece this chapter has built together into one final, complete exercise.


### 2. Internal Working — Step by Step

**Running a genuine A/B test between two retrieval strategies, step by step:**

1. **Define the two configurations being compared, differing in exactly one variable** — for this topic's exercise, BM25-only retrieval (Chapter 7 Topic 1's approach) versus a simple hybrid hint (combining BM25 with a query-expansion step, Chapter 9 Topic 2's Hinglish term-expansion technique) — chosen specifically because this project has already measured, real evidence (Chapter 9 Topic 2's own executed code) that term expansion measurably changes BM25's scoring behavior for vocabulary-mismatched queries.
2. **Run Topic 2's exact eval set through both configurations**, holding every other pipeline component constant — the same knowledge base, the same eval-set questions, the same reference answers — so that any measured difference in RAGAS scores (Topic 1's metrics, computed via Topic 3's methodology) can be attributed specifically to the one variable that actually changed between the two configurations.
3. **Compare context precision and context recall specifically** — the two retrieval-layer RAGAS metrics — since this is a retrieval-strategy comparison, not a generation-strategy one; faithfulness and answer relevancy would only be relevant if the generation step itself were also being varied between the two configurations, which it isn't in this specific test.
4. **Determine statistical or practical significance, not just a raw difference** — given Topic 2's necessarily small eval set, a small measured difference between two configurations' scores could easily be noise rather than a genuine, reliable improvement — exactly the same small-sample-size caution this notebook series has raised repeatedly, now specifically applied to interpreting an A/B test's result rather than treating any measured difference as automatically meaningful.


### 3. How This Is Implemented in This Project

- This notebook's A/B test compares BM25-only retrieval against BM25-plus-Hinglish-term-expansion (Chapter 9 Topic 2's real, already-tested technique), reusing Topic 2's real eval set and Topic 3's real metric-computation methodology — every piece of infrastructure this specific comparison needs was already built in this chapter's earlier topics, and this topic's contribution is combining them into the actual comparison.
- Given this notebook's real eval set (Topic 2) contains genuine, already-somewhat-English-leaning extracted questions (a limitation Topic 2 explicitly acknowledged — no deliberate Hinglish stratification), this topic's exercise additionally constructs a few explicitly Hinglish-phrased test queries, directly reusing Chapter 9 Topic 2's own real Hinglish term mapping, to ensure the comparison actually exercises the specific scenario (vocabulary mismatch) the term-expansion technique is designed to address — otherwise the A/B test would risk measuring no difference simply because neither configuration's advantage was ever actually exercised by the test queries used.
- This chapter's complete arc culminates here: Topic 1's validated metrics, Topic 2's real eval set (extended with Hinglish queries for this specific test), and Topic 3's proven run-and-aggregate methodology combine into a genuine, evidence-based answer to a real, practical question — does Chapter 9 Topic 2's term-expansion technique actually improve retrieval quality, measured properly, rather than just plausibly?


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **An A/B test's validity depends entirely on genuinely holding every variable but one constant** — if the two configurations being compared differ in more than the one variable under test (a different knowledge base version, a different eval-set sample), any measured difference can't be confidently attributed to the actual variable of interest, undermining the entire comparison's validity, regardless of how carefully the metrics themselves were computed.
- **A test query set that doesn't actually exercise the scenario the compared techniques differ on produces a falsely inconclusive result** — exactly the risk this topic's theory explicitly addresses by adding genuinely Hinglish test queries specifically, since Topic 2's original eval set, being somewhat English-leaning, might otherwise show no measurable difference between the two configurations simply because neither configuration's actual advantage was ever exercised by the test data used.
- **This topic's small eval set (inherited from Topic 2) makes any measured difference genuinely uncertain, not a definitive verdict** — a difference of a few percentage points on a handful of examples could easily reverse with a different, equally-small sample; this notebook's own results should be read as an illustration of the A/B testing methodology, not a final, confident verdict on which retrieval strategy is actually better at real production scale.
- **Debugging an unexpected or counter-intuitive A/B test result requires inspecting specific per-query results** (Topic 3's per-entry inspection discipline), not just trusting the aggregate comparison — a configuration that scores worse in aggregate might still be better for a specific query subtype, information only per-query inspection reveals.
- **Monitoring:** if this A/B testing methodology were extended into genuine, ongoing production practice, each new retrieval-strategy proposal (a new chunking approach, Chapter 9 Topic 10; a new embedding model, Chapter 7 Topic 2) would go through this same controlled-comparison process before being adopted, turning this topic's one-time exercise into a repeatable organizational practice, exactly mirroring Chapter 10 Topic 7's regression-testing discipline applied specifically to retrieval-strategy decisions.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Comparing on this project's real eval set (Topic 2, extended with Hinglish queries) vs a purely synthetic test set constructed just for this comparison:** using real, project-representative data (this topic's approach) ensures the comparison reflects genuine production query patterns, at the cost of the eval set's own limitations (Topic 2's explicitly stated small size and construction-method caveats) — a reasonable trade-off given real data's representativeness advantage, provided the eval set's limitations are stated honestly rather than papered over.
- **Testing one variable at a time (BM25 vs BM25-plus-expansion) vs testing several variables simultaneously:** isolating one variable (this topic's approach) produces a cleaner, more directly attributable result, at the cost of needing multiple separate A/B tests to evaluate multiple candidate improvements — the right default whenever a clean, confident attribution is more valuable than testing efficiency, which is generally the case for architectural decisions with real, ongoing consequences.
- **How much measured difference should be considered "significant enough to adopt" the winning configuration:** given this notebook's necessarily small eval set, this question can't be answered with real statistical confidence here — a genuine production decision would need either a considerably larger eval set or a formal statistical significance test, neither of which this notebook's small, illustrative exercise can properly support.


### 6. Alternatives and When to Use Each

- **A/B testing on a real, representative eval set (this topic's approach, extended with deliberately-included Hinglish queries):** the right default whenever comparing two genuinely different retrieval configurations, provided the eval set actually exercises the scenario the configurations are expected to differ on.
- **A/B testing on a purely synthetic, narrowly-constructed test set:** appropriate specifically for validating a targeted, well-understood hypothesis (like Chapter 9 Topic 8's Smart Saver FD test) where a clean, guaranteed-to-exercise-the-scenario test case matters more than broad representativeness.
- **Skipping controlled comparison entirely, adopting a new retrieval strategy based on its general reputation or theoretical appeal:** the approach this notebook series has repeatedly warned against — exactly the mistake evidence-based adoption discipline exists to prevent, across MCP (Chapter 11 Topic 3), chunking strategies (Chapter 9 Topic 10), and now retrieval-strategy choices specifically.


### 7. Common Mistakes and Production Failures

- Changing more than one variable between the two configurations being compared, making it impossible to confidently attribute any measured difference to a specific, actual cause.
- Testing on an eval set that doesn't actually exercise the scenario the compared configurations are expected to differ on, producing a falsely inconclusive or misleading result.
- Treating a small, measured difference on a necessarily small eval set as a definitive, confident verdict, rather than recognizing the same small-sample-size uncertainty this notebook series has raised repeatedly.
- Comparing only aggregate scores without inspecting specific per-query results, missing cases where one configuration might be better for a specific query subtype even if it scores worse in aggregate overall.
- Adopting a new retrieval strategy based on general reputation or intuition rather than this kind of controlled, measured comparison, reproducing the exact mistake this entire chapter's evidence-based methodology exists to prevent.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What makes a retrieval-strategy comparison a genuine A/B test, rather than just running two configurations and comparing whatever numbers come out?
  A: Holding every variable but one constant — the same eval set, the same knowledge base, the same reference answers — so that any measured difference in scores can be confidently attributed to the one variable that actually changed between the two configurations. Without this control, a measured difference could stem from any number of uncontrolled factors, not the specific change being evaluated.

- Q: Why does this topic's A/B test focus specifically on context precision and context recall, rather than all four RAGAS metrics?
  A: This is specifically a retrieval-strategy comparison — the two configurations differ only in how retrieval is performed, not in generation. Context precision and context recall are the two RAGAS metrics that evaluate retrieval quality specifically; faithfulness and answer relevancy evaluate generation, which isn't the variable being tested here.

**Intermediate**

- Q: Why did this topic's exercise deliberately add Hinglish test queries beyond Topic 2's original eval set, rather than using that eval set unchanged?
  A: Topic 2's eval set, drawn from real emails, happened to lean somewhat English in its extracted questions, and didn't deliberately include Hinglish-heavy content. Since the comparison specifically tests a Hinglish-vocabulary-mismatch-targeting technique (Chapter 9 Topic 2's term expansion), using only the original eval set risked the comparison showing no measurable difference simply because neither configuration's actual advantage was ever exercised by the test data — the added Hinglish queries ensure the test actually exercises the scenario the compared techniques are expected to differ on.

- Q: Why is a measured difference on this notebook's small eval set not sufficient grounds for a confident, final production decision?
  A: A difference of a few percentage points measured on a handful of examples could easily reverse with a different, equally-small sample — exactly the same small-sample-size instability this notebook series has raised repeatedly (Chapter 7 Topic 9, Chapter 10 Topic 7). A genuine production decision needs either a considerably larger eval set or formal statistical significance testing to distinguish a real, reliable improvement from measurement noise.

**Advanced**

- Q: Design a complete, ongoing A/B testing practice for this project's future retrieval-strategy decisions, building on this topic's one-time exercise.
  A: Every proposed retrieval-strategy change (a new chunking approach, a different embedding model, a new query-transformation technique) should go through the same controlled-comparison process demonstrated in this topic: hold every other variable constant, run the same, sufficiently large and representative eval set (extended, as needed, with queries specifically designed to exercise whatever scenario the proposed change targets) through both the current and proposed configurations, compare the relevant RAGAS metrics (context precision/recall for retrieval changes, faithfulness/relevancy for generation changes), and require a measured, statistically meaningful improvement before adopting the change — turning this topic's one-time illustrative exercise into Chapter 10 Topic 7's regression-testing discipline, applied specifically to retrieval-strategy evolution over time.

- Q: A teammate reports that a new retrieval configuration shows a higher aggregate context precision score in an A/B test, and proposes adopting it immediately. What would you want to check first?
  A: First, confirm the comparison genuinely held every other variable constant — no incidental differences beyond the one actual retrieval-strategy change. Second, check whether the eval set used actually contains enough examples, and specifically enough examples of the scenario the new configuration is meant to improve, to trust the measured difference rather than treating it as small-sample noise. Third, inspect specific per-query results rather than only the aggregate — a configuration with a higher aggregate score could still perform worse on some specific, important query subtype, information the aggregate number alone wouldn't reveal.

**Scenario-based**

- Q: This topic's A/B test shows BM25-plus-expansion scoring meaningfully higher context precision than BM25-only, but only on the deliberately-added Hinglish queries, with no difference on the original, more English-leaning eval-set entries. How would you interpret and act on this?
  A: This is exactly the expected, correct result if term expansion is a targeted fix for vocabulary mismatch specifically, rather than a general retrieval-quality improvement — it should help precisely the queries it's designed to help, and show no effect where its specific mechanism doesn't apply. This segmented result is more informative and more actionable than a single blended aggregate score would have been (echoing Chapter 9 Topic 7's segmentation discipline), and supports adopting term expansion specifically for its intended purpose, rather than either over-generalizing its benefit to all queries or dismissing it because its aggregate effect across the whole eval set looks modest.

**Follow-up questions to expect:**

- "How would you decide when your eval set is large enough to run a formal statistical significance test rather than just comparing raw numbers?"
- "What would you do if two different metrics (context precision and context recall) disagreed about which configuration is better?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic is the culmination of a controlled-comparison discipline that has recurred throughout this entire notebook series in many different specific forms** — recognizing A/B testing as the general pattern underlying Chapter 7 Topic 9's retrieval evaluation, Chapter 9 Topic 10's chunking comparison, Chapter 10 Topic 4's schema comparison, and Chapter 10 Topic 7's regression testing is a mark of genuinely transferable understanding, connecting what might otherwise seem like several separate, topic-specific techniques into one coherent, reusable methodology.
- **The principle of ensuring a test set actually exercises the scenario under comparison is a general experimental design concern**, not unique to retrieval-strategy A/B testing — any controlled comparison risks a false negative (no measured difference) if the test conditions never actually engage the mechanism being compared, a concern recurring throughout experimental design generally, well beyond this specific application.
- **This topic closes Chapter 14's complete arc**: Topic 1 built the metrics, Topic 2 built the eval set, Topic 3 proved the run-and-aggregate methodology, Topic 4 built the tracing infrastructure for detailed inspection, and this topic finally applies all of it to answer a genuine, practical question with real, measured evidence — completing the chapter's progression from understanding individual pieces to using them together for an actual, evidence-based decision.

### 10. Quick Revision Summary (for last-minute interview prep)

> A/B testing retrieval strategies means holding every variable but one constant between two configurations, running the same real eval set (Topic 2, extended here with deliberately-included Hinglish queries to ensure the comparison actually exercises the scenario under test) through both, and comparing the relevant RAGAS metrics (context precision and context recall specifically, since this is a retrieval-layer comparison) using Topic 3's proven run-and-aggregate methodology. This topic is the direct culmination of a controlled-comparison discipline recurring throughout this entire notebook series — Chapter 7 Topic 9's retrieval evaluation, Chapter 9 Topic 10's chunking comparison, Chapter 10 Topic 4's schema comparison, and Chapter 10 Topic 7's regression testing were all, at their core, the same underlying methodology applied to different specific decisions. A test set that doesn't actually exercise the scenario two configurations are expected to differ on risks a falsely inconclusive result, which is precisely why this topic deliberately adds Hinglish queries rather than relying on Topic 2's somewhat English-leaning original eval set alone. Given this notebook's necessarily small eval set, any measured difference should be read as illustrating the methodology, not as a statistically confident, final production verdict — a genuine production decision needs either a considerably larger eval set or formal significance testing before a winning configuration is confidently adopted.


### Module 1: Setup — Two Retrieval Configurations, One Variable Difference

Define BM25-only vs BM25-plus-Hinglish-expansion (Chapter 9 Topic 2's real technique), differing in EXACTLY one variable, and build the test set including deliberately-added Hinglish queries.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Two configurations, ONE variable difference
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return text.lower().split()

# this project's real knowledge base content, unchanged across BOTH configs
KNOWLEDGE_BASE = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
    "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.",
]
tokenized_corpus = [tokenize(doc) for doc in KNOWLEDGE_BASE]
bm25_index = BM25Okapi(tokenized_corpus)

# Chapter 9 Topic 2's REAL Hinglish term mapping, reused unchanged
HINGLISH_TO_ENGLISH = {
    "byaj": "interest",
    "nikalna": "withdrawal",
}

def expand_query(query: str, term_map: dict) -> str:
    """EXACT SAME function from Chapter 9 Topic 2, unchanged."""
    tokens = tokenize(query)
    expanded = list(tokens)
    for token in tokens:
        if token in term_map and term_map[token] not in expanded:
            expanded.append(term_map[token])
    return " ".join(expanded)


def retrieve_config_A(query: str, top_k: int = 2) -> list:
    """CONFIGURATION A: BM25-only, no query transformation. This is
    the BASELINE -- everything else in this test is held constant
    relative to this configuration."""
    scores = bm25_index.get_scores(tokenize(query))
    ranked = sorted(range(len(KNOWLEDGE_BASE)), key=lambda i: scores[i], reverse=True)
    return [KNOWLEDGE_BASE[i] for i in ranked[:top_k]]


def retrieve_config_B(query: str, top_k: int = 2) -> list:
    """CONFIGURATION B: BM25 + Hinglish term expansion. The ONLY
    variable that differs from Configuration A is this expansion step
    -- same knowledge base, same BM25 index, same top_k."""
    expanded_query = expand_query(query, HINGLISH_TO_ENGLISH)
    scores = bm25_index.get_scores(tokenize(expanded_query))
    ranked = sorted(range(len(KNOWLEDGE_BASE)), key=lambda i: scores[i], reverse=True)
    return [KNOWLEDGE_BASE[i] for i in ranked[:top_k]]


# Topic 2's original (English-leaning) eval set entries
ORIGINAL_EVAL_ENTRIES = [
    {"question": "What is the penalty for premature FD withdrawal?", "type": "english",
     "expected_topic_keyword": "penalty"},
    {"question": "Do senior citizens get extra interest on FDs?", "type": "english",
     "expected_topic_keyword": "senior"},
]

# DELIBERATELY ADDED Hinglish queries -- ensuring this A/B test actually
# exercises the specific scenario Configuration B is designed to help with,
# per this topic's own theory (avoiding a falsely inconclusive test).
# Each includes an "intent_question" -- the English semantic equivalent a
# REAL LLM judge would understand even reading the Hinglish directly; our
# offline word-overlap judge proxy cannot understand Hinglish semantically,
# so we use this explicit equivalent for the JUDGING step specifically,
# while retrieval itself still only ever sees the raw Hinglish query.
HINGLISH_EVAL_ENTRIES = [
    {"question": "mera byaj kitna milega", "type": "hinglish",
     "expected_topic_keyword": "senior"},   # "byaj" (interest) -- best real KB match is the senior-citizen interest chunk
    {"question": "jaldi nikalna hai paisa", "type": "hinglish",
     "expected_topic_keyword": "penalty"},  # "nikalna" (withdraw) -- best real KB match is the withdrawal penalty chunk
]

FULL_EVAL_SET = ORIGINAL_EVAL_ENTRIES + HINGLISH_EVAL_ENTRIES

print("=" * 70)
print("MODULE 1: TWO CONFIGURATIONS, ONE VARIABLE DIFFERENCE")
print("=" * 70)
print(f"\nConfiguration A: BM25-only (baseline)")
print(f"Configuration B: BM25 + Hinglish term expansion (Chapter 9 Topic 2)")
print(f"\nFull eval set: {len(FULL_EVAL_SET)} entries "
      f"({len(ORIGINAL_EVAL_ENTRIES)} original English-leaning + "
      f"{len(HINGLISH_EVAL_ENTRIES)} deliberately-added Hinglish)")

for entry in FULL_EVAL_SET:
    entry_type = entry["type"]
    entry_question = entry["question"]
    print(f"  [{entry_type}] '{entry_question}'")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: TWO CONFIGURATIONS, ONE VARIABLE DIFFERENCE

Configuration A: BM25-only (baseline)
Configuration B: BM25 + Hinglish term expansion (Chapter 9 Topic 2)

Full eval set: 4 entries (2 original English-leaning + 2 deliberately-added Hinglish)
  [english] 'What is the penalty for premature FD withdrawal?'
  [english] 'Do senior citizens get extra interest on FDs?'
  [hinglish] 'mera byaj kitna milega'
  [hinglish] 'jaldi nikalna hai paisa'

Module 1 complete. Run Module 2 next.


### Module 2: Running Both Configurations Across the Full Eval Set — Context Precision

Compute context precision (Topic 1's real formula) for BOTH configurations across every eval entry, reporting per-segment results separately -- never a single blended aggregate.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Context precision, BOTH configs, per-segment reporting
# ------------------------------------------------------------------

def simulate_entailment_judgment(claim: str, context: str) -> bool:
    claim_words = set(w for w in claim.lower().split() if len(w) > 3)
    context_words = set(w for w in context.lower().split() if len(w) > 3)
    if not claim_words:
        return True
    overlap_ratio = len(claim_words & context_words) / len(claim_words)
    return overlap_ratio >= 0.5

def compute_context_precision(question: str, retrieved_chunks: list) -> float:
    """Topic 1's REAL formula, unchanged."""
    relevance_flags = [simulate_entailment_judgment(question, c) for c in retrieved_chunks]
    precisions_at_k = []
    relevant_so_far = 0
    for k, is_relevant in enumerate(relevance_flags, start=1):
        if is_relevant:
            relevant_so_far += 1
            precisions_at_k.append(relevant_so_far / k)
    return sum(precisions_at_k) / len(precisions_at_k) if precisions_at_k else 0.0


print("=" * 70)
print("MODULE 2: CONTEXT PRECISION -- CONFIG A vs CONFIG B, PER ENTRY")
print("=" * 70)

results_A = []
results_B = []

def compute_context_precision_by_keyword(expected_keyword: str, retrieved_chunks: list) -> float:
    """A more robust relevance judge for this specific test: checks
    whether each retrieved chunk actually discusses the expected real
    topic (a keyword a human labeler determined is the correct topic
    for this question) -- standing in for what a real LLM judge would
    understand semantically, avoiding the fragility of raw word-overlap
    against a short, paraphrased intent question."""
    relevance_flags = [expected_keyword in chunk.lower() for chunk in retrieved_chunks]
    precisions_at_k = []
    relevant_so_far = 0
    for k, is_relevant in enumerate(relevance_flags, start=1):
        if is_relevant:
            relevant_so_far += 1
            precisions_at_k.append(relevant_so_far / k)
    return sum(precisions_at_k) / len(precisions_at_k) if precisions_at_k else 0.0


for entry in FULL_EVAL_SET:
    question = entry["question"]
    expected_keyword = entry["expected_topic_keyword"]

    retrieved_A = retrieve_config_A(question)
    precision_A = compute_context_precision_by_keyword(expected_keyword, retrieved_A)
    results_A.append({"question": question, "type": entry["type"], "precision": precision_A})

    retrieved_B = retrieve_config_B(question)
    precision_B = compute_context_precision_by_keyword(expected_keyword, retrieved_B)
    results_B.append({"question": question, "type": entry["type"], "precision": precision_B})

    entry_type = entry["type"]
    print(f"\n[{entry_type}] '{question}' (expected topic keyword: '{expected_keyword}')")
    print(f"  Config A (BM25-only) retrieved: '{retrieved_A[0][:55]}...'")
    print(f"    context_precision = {precision_A:.3f}")
    print(f"  Config B (BM25+expansion) retrieved: '{retrieved_B[0][:55]}...'")
    print(f"    context_precision = {precision_B:.3f}")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: CONTEXT PRECISION -- CONFIG A vs CONFIG B, PER ENTRY

[english] 'What is the penalty for premature FD withdrawal?' (expected topic keyword: 'penalty')
  Config A (BM25-only) retrieved: 'Premature withdrawal of FD incurs a 1 percent penalty o...'
    context_precision = 1.000
  Config B (BM25+expansion) retrieved: 'Premature withdrawal of FD incurs a 1 percent penalty o...'
    context_precision = 1.000

[english] 'Do senior citizens get extra interest on FDs?' (expected topic keyword: 'senior')
  Config A (BM25-only) retrieved: 'Senior citizens receive an additional 0.5 percent inter...'
    context_precision = 1.000
  Config B (BM25+expansion) retrieved: 'Senior citizens receive an additional 0.5 percent inter...'
    context_precision = 1.000

[hinglish] 'mera byaj kitna milega' (expected topic keyword: 'senior')
  Config A (BM25-only) retrieved: 'Premature withdrawal of FD incurs a 1 percent penalty o...'
    context_precision = 0.500
  Config B (BM25+expansion) retrieved:

### Module 3: Segmented Comparison — The Result That Matters

Aggregate SEPARATELY for English vs Hinglish segments, revealing whether the expansion technique helps precisely where it should, per this topic's segmentation discipline.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Segmented A/B comparison -- English vs Hinglish, separately
# ------------------------------------------------------------------

def aggregate_by_segment(results: list, segment_type: str) -> float:
    segment_scores = [r["precision"] for r in results if r["type"] == segment_type]
    return sum(segment_scores) / len(segment_scores) if segment_scores else 0.0


print("=" * 70)
print("MODULE 3: SEGMENTED A/B COMPARISON (never a single blended aggregate)")
print("=" * 70)

for segment in ["english", "hinglish"]:
    avg_A = aggregate_by_segment(results_A, segment)
    avg_B = aggregate_by_segment(results_B, segment)
    difference = avg_B - avg_A

    print(f"\n[{segment.upper()} segment]")
    print(f"  Config A (BM25-only) average context precision:      {avg_A:.3f}")
    print(f"  Config B (BM25+expansion) average context precision: {avg_B:.3f}")
    print(f"  Difference (B - A): {difference:+.3f}")

overall_avg_A = sum(r["precision"] for r in results_A) / len(results_A)
overall_avg_B = sum(r["precision"] for r in results_B) / len(results_B)

print(f"\n[OVERALL, BLENDED aggregate -- shown ONLY to demonstrate why this")
print(f" is LESS informative than the segmented view above]")
print(f"  Config A overall average: {overall_avg_A:.3f}")
print(f"  Config B overall average: {overall_avg_B:.3f}")

hinglish_diff = aggregate_by_segment(results_B, "hinglish") - aggregate_by_segment(results_A, "hinglish")
english_diff = aggregate_by_segment(results_B, "english") - aggregate_by_segment(results_A, "english")

print(f"\nCONCLUSION:")
if hinglish_diff > 0.01 and abs(english_diff) < 0.01:
    print(f"  Config B (term expansion) shows a MEASURABLE improvement SPECIFICALLY")
    print(f"  on Hinglish queries ({hinglish_diff:+.3f}), with NO meaningful change on")
    print(f"  English queries ({english_diff:+.3f}) -- EXACTLY the theory's predicted,")
    print(f"  targeted result: a technique that helps precisely where it should,")
    print(f"  with no effect where its mechanism doesn't apply. This segmented")
    print(f"  result is far more informative and actionable than the blended")
    print(f"  overall average above, which would have understated Config B's")
    print(f"  genuine, targeted value.")
else:
    print(f"  Results did not show the expected clean segment-specific pattern in")
    print(f"  this run -- worth inspecting individual entries directly (Module 2's")
    print(f"  output) rather than trusting only this aggregate summary.")

print(f"\nHONEST CAVEAT: this A/B test used a small, illustrative eval set")
print(f"({len(FULL_EVAL_SET)} entries) -- this demonstrates the A/B TESTING")
print(f"METHODOLOGY correctly, but is NOT a statistically confident, final")
print(f"production verdict. A real decision needs a considerably larger,")
print(f"more representative eval set.")

print("\nModule 3 complete. All Chapter 14 Topic 5 modules done.")
print()
print("=" * 70)
print("CHAPTER 14 TOPIC 5 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  TWO configurations differing in EXACTLY ONE variable (term expansion)
  -- everything else (knowledge base, BM25 index, top_k) held constant.

  Eval set DELIBERATELY includes Hinglish queries alongside the original
  English-leaning set -- ensuring the test actually exercises the
  scenario the compared technique is designed to address.

  SEGMENTED reporting (English vs Hinglish, separately) reveals a
  targeted, mechanism-consistent effect that a single BLENDED aggregate
  would have understated or hidden entirely -- demonstrated concretely.

  This topic combines EVERY piece this chapter built: Topic 1's metrics,
  Topic 2's eval-set-construction discipline, Topic 3's run-and-aggregate
  methodology -- into one final, evidence-based comparison.

  Small eval set = illustrative methodology, NOT a statistically
  confident final production verdict -- always stated honestly.
""")


MODULE 3: SEGMENTED A/B COMPARISON (never a single blended aggregate)

[ENGLISH segment]
  Config A (BM25-only) average context precision:      1.000
  Config B (BM25+expansion) average context precision: 1.000
  Difference (B - A): +0.000

[HINGLISH segment]
  Config A (BM25-only) average context precision:      0.750
  Config B (BM25+expansion) average context precision: 1.000
  Difference (B - A): +0.250

[OVERALL, BLENDED aggregate -- shown ONLY to demonstrate why this
 is LESS informative than the segmented view above]
  Config A overall average: 0.875
  Config B overall average: 1.000

CONCLUSION:
  Config B (term expansion) shows a MEASURABLE improvement SPECIFICALLY
  on Hinglish queries (+0.250), with NO meaningful change on
  English queries (+0.000) -- EXACTLY the theory's predicted,
  targeted result: a technique that helps precisely where it should,
  with no effect where its mechanism doesn't apply. This segmented
  result is far more informative and actionable than the b